In [13]:
import json
import pandas as pd

In [14]:
# open raw JSON

with open("../data/raw/estat_sdg_05_20_filtered_en.json", "r") as f:
    data = json.load(f)

In [15]:
# to get actual values

geo_labels = data["dimension"]["geo"]["category"]["label"]
time_labels = data["dimension"]["time"]["category"]["label"]
values = data["value"]

geo_list = list(geo_labels.values())
time_list = list(time_labels.values())

In [16]:
# turns the raw JSON into a normal table

rows = []

for key, value in values.items():
    index = int(key)
    geo_index = index // len(time_list)
    time_index = index % len(time_list)

    country = geo_list[geo_index]
    year = time_list[time_index]

    rows.append([country, year, value])

df_json = pd.DataFrame(rows, columns=["country", "year", "pay_gap"])
df_json.head()

,country,year,pay_gap
0,Albania,2018,6.8
1,Austria,2006,25.5
2,Austria,2007,25.5
3,Austria,2008,25.1
4,Austria,2009,24.3


In [17]:
# standardize column names and data types

df_json.columns = df_json.columns.str.strip().str.lower()
df_json["year"] = pd.to_numeric(df_json["year"], errors="coerce")
df_json["pay_gap"] = pd.to_numeric(df_json["pay_gap"], errors="coerce")

In [18]:
# JSON data contains a lot of status codes. This standardize the status codes into readable words.

status_labels = data["extension"]["status"]["label"]
status_raw = data["status"]

status_rows = []

for key, status_code in status_raw.items():
    index = int(key)
    geo_index = index // len(time_list)
    time_index = index % len(time_list)

    country = geo_list[geo_index]
    year = time_list[time_index]
    status_text = status_labels.get(status_code, status_code)

    status_rows.append([country, year, status_text])

df_status = pd.DataFrame(status_rows, columns=["country", "year", "status"])
df_status.head()

,country,year,status
0,Austria,2023,provisional
1,Austria,2024,provisional
2,Belgium,2023,provisional
3,Belgium,2024,provisional
4,Bulgaria,2023,provisional


In [21]:
# before adding readable status to the table, both columns must be the same type 
# (before: df_json, year is a number (int64), and df_status, year is a text (object))

df_status["year"] = pd.to_numeric(df_status["year"], errors="coerce")

In [22]:
df_json = df_json.merge(df_status, on=["country", "year"], how="left")
df_json.head()

,country,year,pay_gap,status
0,Albania,2018,6.8,NaN
1,Austria,2006,25.5,NaN
2,Austria,2007,25.5,NaN
3,Austria,2008,25.1,NaN
4,Austria,2009,24.3,NaN


In [23]:
not_countries = [
    "European Union - 27 countries (from 2020)",
    "Euro area – 21 countries (from 2026)",
    "Euro area – 20 countries (2023-2025)",
    "Euro area - 19 countries  (2015-2022)",
    "European Union - 27 countries (2007-2013)"
]

# remove region-level rows
df_json = df_json[~df_json["country"].isin(not_countries)]

# remove missing values
df_json = df_json.dropna(subset=["pay_gap"])

# remove duplicates
df_json = df_json.drop_duplicates()

# sort the data neatly
df_json = df_json.sort_values(["country", "year"]).reset_index(drop=True)

df_json.head()

,country,year,pay_gap,status
0,Albania,2018,6.8,NaN
1,Austria,2006,25.5,NaN
2,Austria,2007,25.5,NaN
3,Austria,2008,25.1,NaN
4,Austria,2009,24.3,NaN


In [24]:
# make the missing status easier to read 
df_json["status"] = df_json["status"].fillna("no special flag")
df_json.head()

,country,year,pay_gap,status
0,Albania,2018,6.8,no special flag
1,Austria,2006,25.5,no special flag
2,Austria,2007,25.5,no special flag
3,Austria,2008,25.1,no special flag
4,Austria,2009,24.3,no special flag


In [25]:
# inspect the cleaned table

# shows the column types
df_json.info()

# shows missing values
df_json.isna().sum()

# checks duplicate rows
df_json.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 585 entries, 0 to 584
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   country  585 non-null    object 
 1   year     585 non-null    int64  
 2   pay_gap  585 non-null    float64
 3   status   585 non-null    object 
dtypes: float64(1), int64(1), object(2)
memory usage: 18.4+ KB


np.int64(0)

The dataset had 585 complete rows with no missing values in the main columns. The year and pay gap variables were stored as numeric types and there's no duplicate rows, which means the data was ready for analysis.

In [28]:
# checks final cleaned data
df_json.head(10)

,country,year,pay_gap,status
0,Albania,2018,6.8,no special flag
1,Austria,2006,25.5,no special flag
2,Austria,2007,25.5,no special flag
3,Austria,2008,25.1,no special flag
4,Austria,2009,24.3,no special flag
5,Austria,2010,24.0,no special flag
6,Austria,2011,23.5,no special flag
7,Austria,2012,22.9,no special flag
8,Austria,2013,22.3,no special flag
9,Austria,2014,22.2,no special flag


In [32]:
df_json["country_type"] = df_json["country"].apply(
    lambda x: "aggregate_region" if "Euro area" in x or "European Union" in x else "country"
)

df_json.head()

,country,year,pay_gap,status,country_type
0,Albania,2018,6.8,no special flag,country
1,Austria,2006,25.5,no special flag,country
2,Austria,2007,25.5,no special flag,country
3,Austria,2008,25.1,no special flag,country
4,Austria,2009,24.3,no special flag,country


To make it clearer, we added a label to each row to show whether it represents one actual country or a larger region/group.